# 01. Khảo sát dữ liệu (ISOT)
Đặt `True.csv` và `Fake.csv` vào `data/raw/isot/` (xem `docs/khao_sat_du_lieu.md`). Nếu chưa có dữ liệu, ô đầu sẽ báo và dừng.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
sys.path.insert(0, str(Path('..') / 'src'))
from fakenews.preprocess import clean_text

root = Path('..') / 'data' / 'raw' / 'isot'
if not (root / 'True.csv').exists() or not (root / 'Fake.csv').exists():
    raise SystemExit('Chưa có dữ liệu ISOT trong data/raw/isot/. Xem docs/khao_sat_du_lieu.md để tải.')

true_df = pd.read_csv(root / 'True.csv'); true_df['label'] = 0
fake_df = pd.read_csv(root / 'Fake.csv'); fake_df['label'] = 1
df = pd.concat([true_df, fake_df], ignore_index=True)
print(df.shape)
df.head()

In [ ]:
print(df['label'].value_counts().rename({0: 'tin thật', 1: 'tin giả'}))
print('\nCác cột:', list(df.columns))
print('\nGiá trị thiếu:\n', df.isna().sum())

In [ ]:
df['text_clean'] = df['text'].map(clean_text)
df['n_words'] = df['text_clean'].str.split().str.len()
print(df.groupby('label')['n_words'].describe())
print('\nSố bài trùng nội dung:', df.duplicated('text_clean').sum())
print('Số bài rỗng/quá ngắn (<20 từ):', (df['n_words'] < 20).sum())

## Kiểm tra rò rỉ nhãn
Nếu một dấu hiệu xuất hiện gần như chỉ ở một lớp (ví dụ tên hãng tin ở đầu bài, cột `subject`), mô hình có thể học dấu hiệu đó thay vì học nội dung.

In [ ]:
has_reuters = df['text'].str.contains(r'\(Reuters\)', regex=True, na=False)
print('Tỷ lệ bài có "(Reuters)" theo lớp:')
print(has_reuters.groupby(df['label']).mean().rename({0: 'tin thật', 1: 'tin giả'}))
if 'subject' in df.columns:
    print('\nPhân bố cột subject theo lớp:')
    print(pd.crosstab(df['subject'], df['label']))

## Ghi chú rút ra
- (điền sau khi chạy) Dấu hiệu rò rỉ phát hiện được:
- (điền) Cách xử lý đã chọn: